# **Preprocessing Data**
**Data Resources :**
1. **Kaggle :** Chennai Water Reservoirs (`Chennai Metropolitan Water Supply & sewage Board`). Hydrological Data is obtained from this side. (water level, inflow, outflow, rain water, etc.)

2. **NASA Power DAV :** Weather Data or Climate Data sources from Here. (Temperature, wind speed, etc.)


In [ ]:
import pandas as pd
import numpy as np
import os
import sys

###Configuration for Google Colab

Mount to Drive and fetch data from google drive

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/EDI-III Project/Data'
else:
    BASE_PATH = '.'

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Defining Paths for Kaggle and NASA directories in Raw Data.

In [ ]:
KAGGLE_DIR = os.path.join(BASE_PATH, 'Raw Data','Kaggel')
NASA_DIR = os.path.join(BASE_PATH, 'Raw Data', 'Nasa')

mapping files from both directories to their repective reservoir names

In [ ]:
kaggle_files = {
    'Chembarambakkam': 'chembarambakkam_data.csv',
    'Cholavaram': 'cholavaram_data.csv',
    'Kannakottai': 'kannankottai_thervoy_kandigai_data.csv',
    'Poondi': 'poondi_data.csv',
    'Puzhal': 'puzhal_data.csv',
    'Veeranam': 'veeranam_data.csv'
}

nasa_files = {
    'Chembarambakkam': 'Chembarambakkam.csv',
    'Cholavaram': 'Cholavaram.csv',
    'Kannakottai': 'Kannakottai.csv',
    'Poondi': 'Poondi.csv',
    'Puzhal': 'Puzhal.csv',
    'Veeranam': 'verranam.csv'
}

Complete function for **preprocessing** for Both Directories from **Raw Data**. (*Hydrological Data* , *Weather Data*) and storing processed data in **Processed Data Directory**.

In [ ]:
def process_reservoir_data(name, hydro_filename, weather_filename):
    print(f"\nProcessing {name}...")

    hydro_path = os.path.join(KAGGLE_DIR, hydro_filename)
    weather_path = os.path.join(NASA_DIR, weather_filename)

    if not os.path.exists(hydro_path) or not os.path.exists(weather_path):
        print(f"  Error: File not found for {name}.")
        print(f"  Expected: {hydro_path} \n  and: {weather_path}")
        return None

    try:
        df_hydro = pd.read_csv(hydro_path)
        df_hydro['Date'] = pd.to_datetime(df_hydro['Date'], format='%d-%m-%Y')
        df_hydro = df_hydro.sort_values('Date')
    except Exception as e:
        print(f"  Error loading hydrology data: {e}")
        return None

    try:
        df_weather = pd.read_csv(weather_path, header=16)

        df_weather['Date'] = pd.to_datetime(df_weather[['YEAR', 'MO', 'DY']].astype(str).agg('-'.join, axis=1))

        cols_to_drop = ['IMERG_PRECTOT', 'YEAR', 'MO', 'DY']
        df_weather = df_weather.drop(columns=[c for c in cols_to_drop if c in df_weather.columns])
    except Exception as e:
        print(f"  Error loading weather data: {e}")
        return None

    merged_df = pd.merge(df_hydro, df_weather, on='Date', how='inner')

    merged_df.set_index('Date', inplace=True)

    merged_df['Month'] = merged_df.index.month
    merged_df['DayOfYear'] = merged_df.index.dayofyear

    for lag in [1, 2, 3]:
        merged_df[f'Rainfall_Lag{lag}'] = merged_df['Rainfall (mm)'].shift(lag)
        merged_df[f'Inflow_Lag{lag}'] = merged_df['Inflow (cusecs)'].shift(lag)
        merged_df[f'Storage_Lag{lag}'] = merged_df['Storage (mcft)'].shift(lag)

    merged_df['Rainfall_RollMean_7d'] = merged_df['Rainfall (mm)'].rolling(window=7).mean()
    merged_df['Inflow_RollMean_7d'] = merged_df['Inflow (cusecs)'].rolling(window=7).mean()

    merged_df = merged_df.dropna()

    output_path = os.path.join(BASE_PATH,'Processed Data', f"{name}.csv")
    merged_df.to_csv(output_path)

    print(f"  -> Success! Saved processed file to: {output_path}")
    print(f"  -> Shape: {merged_df.shape}")
    return output_path

if __name__ == "__main__":
    generated_files = []
    print(f"Starting processing in {BASE_PATH}...")

    for name in kaggle_files.keys():
        output_file = process_reservoir_data(name, kaggle_files[name], nasa_files[name])
        if output_file:
            generated_files.append(output_file)

    print("\nProcessing Complete.")
    print("Files created:", generated_files)

Starting processing in /content/drive/MyDrive/EDI-III Project/Data...

Processing Chembarambakkam...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Chembarambakkam.csv
  -> Shape: (7521, 26)

Processing Cholavaram...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Cholavaram.csv
  -> Shape: (7521, 26)

Processing Kannakottai...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Kannakottai.csv
  -> Shape: (7521, 26)

Processing Poondi...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Poondi.csv
  -> Shape: (7521, 26)

Processing Puzhal...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Puzhal.csv
  -> Shape: (7521, 26)

Processing Veeranam...
  -> Success! Saved processed file to: /content/drive/MyDrive/EDI-III Project/Data/Processed Data/Ve